# Model Comparison and Selection


This notebook audits the model selection process for the WNBA salary valuation model.

The main goals are:

1. Compare candidate models using the project-defined KPI framework.
2. Confirm that the selected model is consistent with the primary KPI.
3. Check whether the selected model has overfitting or instability concerns.
4. Evaluate final 2025 holdout performance.
5. Inspect player-level prediction errors.

KPI hierarchy used in this notebook:

- **Primary KPI:** RMSE
- **Secondary KPI:** MAPE
- **Business KPI:** CPWS, we do not examine here
- **Supplementary diagnostics:** MAE, R², train-validation gap, `std_test_score` from tuning results, and player-level prediction errors


In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, root_mean_squared_error

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

#check if we are in the root directory or in the scripts directory
ROOT = Path.cwd()

if not (ROOT / "results").exists():
    ROOT = ROOT.parent

result_dir = ROOT / "results"
artifact_dir = ROOT / "artifacts"

## 1. Load tuning results

The original tuning file is saved as `results/tuning_results.csv`.

For auditing, we create a derived table with additional generalization diagnostics:

- `cv_train_gap = cv_rmse - train_rmse`
- `relative_gap = cv_train_gap / cv_rmse`

These derived columns are not used as the primary selection rule. They are used to audit possible overfitting. `cv_train_gap`: possible overfitting; `relative_gap`: normalized overfitting gap. CV stability is assessed using `std_test_score` from the tuning results.


In [16]:
tuning_path = result_dir / "tuning_results.csv"
tuning_df = pd.read_csv(tuning_path)

tuning_df.head()


,params,std_test_score,rank_test_score,model,cv_rmse,train_rmse
0,"{'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 200}",4171.133047,12,RandomForest,40119.460423,24521.947359
1,"{'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 400}",4168.444043,14,RandomForest,40137.982104,24441.946988
2,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}",3506.819735,4,RandomForest,39956.283417,29423.984091
3,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}",3517.118256,1,RandomForest,39859.578826,29358.021991
4,"{'max_depth': None, 'min_samples_leaf': 8, 'n_estimators': 200}",2915.811025,9,RandomForest,40062.746578,33804.996922


In [30]:
# Add audit diagnostics.
tuning_audit = tuning_df.copy()

tuning_audit["cv_train_gap"] = tuning_audit["cv_rmse"] - tuning_audit["train_rmse"]
tuning_audit["relative_gap"] = tuning_audit["cv_train_gap"] / tuning_audit["cv_rmse"]

# Save a derived audit file without overwriting the original tuning results.
audit_dir = result_dir / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)

audit_path = audit_dir  / "tuning_results_audit.csv"
tuning_audit.to_csv(audit_path, index=False)

print(f"Saved audit tuning results to: {audit_path}")
tuning_audit.head()


Saved audit tuning results to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\audit\tuning_results_audit.csv


,params,std_test_score,rank_test_score,model,cv_rmse,train_rmse,cv_train_gap,relative_gap
0,"{'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 200}",4171.133047,12,RandomForest,40119.460423,24521.947359,15597.513064,0.388777
1,"{'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 400}",4168.444043,14,RandomForest,40137.982104,24441.946988,15696.035116,0.391052
2,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}",3506.819735,4,RandomForest,39956.283417,29423.984091,10532.299326,0.263596
3,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}",3517.118256,1,RandomForest,39859.578826,29358.021991,10501.556834,0.263464
4,"{'max_depth': None, 'min_samples_leaf': 8, 'n_estimators': 200}",2915.811025,9,RandomForest,40062.746578,33804.996922,6257.749656,0.156199


## 2. Audit tuning-stage model selection

The tuning-stage selection should be driven primarily by cross-validated RMSE, since RMSE is our primary predictive KPI.

However, the audit also checks:

- whether the top-ranked models have very similar CV RMSE values;
- whether the selected model has a large train-validation gap (`cv_train_gap`);
- whether CV performance is stable across chronological folds, using `std_test_score`;


In [18]:
# Sort all candidate configurations by CV RMSE.
top_tuning = tuning_audit.sort_values("cv_rmse").head(10)

top_tuning[[
    "model",
    "params",
    "cv_rmse",
    "train_rmse",
    "cv_train_gap",
    "relative_gap",
    "std_test_score",
    "rank_test_score"
]]


,model,params,cv_rmse,train_rmse,cv_train_gap,relative_gap,std_test_score,rank_test_score
3,RandomForest,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}",39859.578826,29358.021991,10501.556834,0.263464,3517.118256,1
15,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 400}",39866.145627,29397.651685,10468.493943,0.262591,3513.784428,2
14,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 200}",39948.695494,29461.503938,10487.191556,0.262516,3513.390167,3
2,RandomForest,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}",39956.283417,29423.984091,10532.299326,0.263596,3506.819735,4
9,RandomForest,"{'max_depth': 6, 'min_samples_leaf': 5, 'n_estimators': 400}",40013.396186,31153.749118,8859.647067,0.221417,3377.753972,5
8,RandomForest,"{'max_depth': 6, 'min_samples_leaf': 5, 'n_estimators': 200}",40027.293482,31204.382549,8822.910933,0.220422,3427.400667,6
17,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 8, 'n_estimators': 400}",40061.927971,33761.242572,6300.685400,0.157274,2911.965364,7
5,RandomForest,"{'max_depth': None, 'min_samples_leaf': 8, 'n_estimators': 400}",40062.456286,33757.522879,6304.933407,0.157378,2911.572362,8
4,RandomForest,"{'max_depth': None, 'min_samples_leaf': 8, 'n_estimators': 200}",40062.746578,33804.996922,6257.749656,0.156199,2915.811025,9
16,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 8, 'n_estimators': 200}",40064.574679,33809.214751,6255.359928,0.156132,2914.318680,10


In [19]:
# Best candidate within each model family.
best_by_model = (
    tuning_audit
    .sort_values("cv_rmse")
    .groupby("model", as_index=False)
    .first()
)

best_by_model[[
    "model",
    "params",
    "cv_rmse",
    "train_rmse",
    "cv_train_gap",
    "relative_gap",
    "std_test_score",
    "rank_test_score"
]]


,model,params,cv_rmse,train_rmse,cv_train_gap,relative_gap,std_test_score,rank_test_score
0,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 3, 'n_estimators': 100}",40139.104494,34112.267306,6026.837188,0.150149,3932.076232,1
1,RandomForest,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}",39859.578826,29358.021991,10501.556834,0.263464,3517.118256,1


### Tuning-stage audit notes

The tuning results show that the selected Random Forest model has the lowest cross-validated RMSE among the tuned candidate models. This is consistent with the project KPI framework, since RMSE is defined as the primary predictive KPI.

The hyperparameter search was intentionally limited to Random Forest and Gradient Boosting. This keeps the optimization process simpler and more reproducible, which is appropriate given the relatively small dataset. However, this also means that the selected Random Forest model should be interpreted as the best model within this tuning scope, not necessarily the best possible model across all model families explored earlier.

The difference between the best Random Forest model and the best Gradient Boosting model is relatively small. The best Random Forest model has a CV RMSE of 39,859.58, while the best Gradient Boosting model has a CV RMSE of 40,139.10. The Random Forest model therefore improves CV RMSE by about 279.53, which is small relative to the overall error scale.

The audit also shows that this small RMSE difference should not be over-interpreted. The best Random Forest model has a `std_test_score` of 3,517.12, while the best Gradient Boosting model has a `std_test_score` of 3,932.08. Since these fold-level variability values are much larger than the CV RMSE difference between the two best models, the exact ranking between the top models is somewhat uncertain.

There is also a trade-off in generalization diagnostics. The selected Random Forest model has a larger train-validation gap, with a `cv_train_gap` of 10,501.56 and a `relative_gap` of 0.263. In comparison, the best Gradient Boosting model has a smaller `cv_train_gap` of 6,026.84 and a `relative_gap` of 0.150. This suggests that Random Forest performs slightly better under the primary KPI, but it may also show more overfitting than Gradient Boosting.

Overall, selecting Random Forest is reasonable under the RMSE-first rule.

## 3. Load final 2025 holdout predictions

The file `results/tuned_predictions.csv` contains player-level predictions from the selected tuned model evaluated on the untouched 2025 holdout set.

Expected columns:

- `player`
- `team`
- `year`
- `group`
- `actual`
- `predicted`
- `residual`
- `abs_error`

Here, `residual = actual - predicted`.

The `residual` column helps identify direction:

- positive residual means the model predicted a higher salary than the player actually earned
- negative residual means the model predicted a lower salary than the player actually earned


In [20]:
pred_path = result_dir / "tuned_predictions.csv"
pred_df = pd.read_csv(pred_path)

pred_df.head()


,player,team,year,group,actual,predicted,residual,abs_error
0,A'ja Wilson,LVA,2025,veteran,200000,214155.701628,14155.701628,14155.701628
1,Napheesa Collier,MIN,2025,unknown,214284,200972.099195,-13311.900805,13311.900805
2,Kelsey Mitchell,IND,2025,veteran,269244,213155.488619,-56088.511381,56088.511381
3,Kelsey Plum,LAS,2025,veteran,202000,214220.125809,12220.125809,12220.125809
4,Paige Bueckers,DAL,2025,rookie,78831,121931.589132,43100.589132,43100.589132


In [21]:
# Basic data checks.
display(pred_df.shape)
display(pred_df.dtypes)
display(pred_df.isna().sum())

# Recompute residual and absolute error to make sure the saved columns are consistent.
pred_df["residual_check"] = pred_df["actual"] - pred_df["predicted"]
pred_df["abs_error_check"] = pred_df["residual_check"].abs()

pred_df[["actual", "predicted", "residual", "residual_check", "abs_error", "abs_error_check"]].head()


(223, 8)

player           str
team             str
year           int64
group            str
actual         int64
predicted    float64
residual     float64
abs_error    float64
dtype: object

player       0
team         0
year         0
group        0
actual       0
predicted    0
residual     0
abs_error    0
dtype: int64

,actual,predicted,residual,residual_check,abs_error,abs_error_check
0,200000,214155.701628,14155.701628,-14155.701628,14155.701628,14155.701628
1,214284,200972.099195,-13311.900805,13311.900805,13311.900805,13311.900805
2,269244,213155.488619,-56088.511381,56088.511381,56088.511381,56088.511381
3,202000,214220.125809,12220.125809,-12220.125809,12220.125809,12220.125809
4,78831,121931.589132,43100.589132,-43100.589132,43100.589132,43100.589132


## 4. Final 2025 holdout KPI evaluation

This section recomputes the final out-of-sample performance metrics from `results/tuned_predictions.csv`. The tuned model was selected during the cross-validation stage, and the 2025 holdout set is used only as the final test set.

Model performance is interpreted according to the project KPI hierarchy:

1. **RMSE** is the primary predictive KPI.
2. **MAPE** is the secondary scale-sensitive KPI.
3. **MAE** and **R²** are supplementary diagnostics only.

Because the project goal is to compare actual salary with model predicted salary, the holdout metrics evaluate how closely the trained model predicts 2025 player salaries. Player-level residuals are then used to inspect model-relative overpayment or underpayment patterns.


In [22]:
y_true = pred_df["actual"]
y_pred = pred_df["predicted"]

metrics = {
    "model": "RandomForestRegressor",
    "train_years": "2021-2024",
    "test_year": 2025,
    "n_train": 742,
    "n_test": len(pred_df),
    "RMSE": root_mean_squared_error(y_true, y_pred),
    "MAE": mean_absolute_error(y_true, y_pred),
    "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    "R2": r2_score(y_true, y_pred),
}

metrics_df = pd.DataFrame([metrics])
metrics_df


,model,train_years,test_year,n_train,n_test,RMSE,MAE,MAPE,R2
0,RandomForestRegressor,2021-2024,2025,742,223,41368.597846,30376.251304,1.287013,0.634203


### KPI interpretation

The final tuned Random Forest model achieves a 2025 holdout RMSE of 41,368.60 and an MAE of 30,376.25. Since RMSE is our primary predictive KPI, the main evaluation result is that the model's prediction error on the 2025 holdout set is about $41K under RMSE. The MAE means that, on average, the model's player-level salary prediction is off by about $30K.

The model also has an R² of 0.634. This means that the model explains a moderate amount of variation in 2025 player salaries compared with a simple mean-prediction baseline. However, R² is only an additional diagnostic metric here. It should not be used as the main reason for selecting the model, because the project prioritizes salary prediction error measured in dollars.

The MAPE is 1.287, or about 128.7%. This value should be interpreted carefully. MAPE divides each absolute error by the player's actual salary, so players with smaller actual salaries can produce very large percentage errors even when the dollar error is not unusually large. Because this notebook reports one overall MAPE rather than separating the evaluation by contract size, the high MAPE may partly reflect the sensitivity of the metric to small salary denominators.

Overall, the final model is reasonable under the RMSE-first rule. Given the small dataset and the instability of aggregate MAPE in this setting, RMSE and MAE are more reliable for the current audit. MAPE should still be reported, but it should not be overemphasized as a model-selection metric without a contract-size-aware analysis. CPWS is not evaluated in this notebook and should be left as future work for the final project presentation.

## 5. Player-level prediction error analysis

This section identifies the largest player-level prediction errors and uses the residual direction to flag players who appear underpaid or overpaid relative to the model-predicted salary.

Since `residual = predicted - actual`:

1. **largest absolute errors** show the largest prediction mistakes;
2. **largest positive residuals** show players whose predicted salary is much higher than their actual salary, so they appear underpaid relative to the model predicted salary;
3. **smallest residuals** show players whose predicted salary is much lower than their actual salary, so they appear overpaid relative to the model predicted salary.

In [23]:
# Make sure residual and absolute error use the same definition as model_testing.ipynb:
# residual = predicted - actual

pred_df = pred_df.copy()

pred_df["residual"] = pred_df["predicted"] - pred_df["actual"]
pred_df["abs_error"] = pred_df["residual"].abs()

pred_df[[
    "player",
    "team",
    "group",
    "actual",
    "predicted",
    "residual",
    "abs_error"
]].head()

,player,team,group,actual,predicted,residual,abs_error
0,A'ja Wilson,LVA,veteran,200000,214155.701628,14155.701628,14155.701628
1,Napheesa Collier,MIN,unknown,214284,200972.099195,-13311.900805,13311.900805
2,Kelsey Mitchell,IND,veteran,269244,213155.488619,-56088.511381,56088.511381
3,Kelsey Plum,LAS,veteran,202000,214220.125809,12220.125809,12220.125809
4,Paige Bueckers,DAL,rookie,78831,121931.589132,43100.589132,43100.589132


In [24]:
# Largest absolute prediction errors.
largest_abs_errors = (
    pred_df
    .sort_values("abs_error", ascending=False)
    .head(10)
)

largest_abs_errors[[
    "player",
    "team",
    "group",
    "actual",
    "predicted",
    "residual",
    "abs_error"
]]


,player,team,group,actual,predicted,residual,abs_error
54,Odyssey Sims,TOT,veteran,7949,138897.386595,130948.386595,130948.386595
53,Odyssey Sims,TOT,veteran,13911,138897.386595,124986.386595,124986.386595
194,Moriah Jefferson,CHI,veteran,145500,22577.850691,-122922.149309,122922.149309
9,Sabrina Ionescu,NYL,rookie,222060,106060.957794,-115999.042206,115999.042206
112,Teaira McCowan,DAL,veteran,201400,89401.316216,-111998.683784,111998.683784
16,Arike Ogunbowale,DAL,unknown,249032,139530.215438,-109501.784562,109501.784562
44,Jewell Loyd,LVA,veteran,249032,140885.526888,-108146.473112,108146.473112
104,Kaila Charles,TOT,veteran,13911,113460.868377,99549.868377,99549.868377
103,Kaila Charles,TOT,veteran,13911,113460.868377,99549.868377,99549.868377
91,Haley Jones,TOT,veteran,36094,128767.144015,92673.144015,92673.144015


In [25]:
# Most underpaid relative to the model-predicted salary:
# predicted salary is much higher than actual salary. The larger the residual, the more underpaid the player is relative to the model's prediction.
most_underpaid_relative_to_model = (
    pred_df
    .sort_values("residual", ascending=False)
    .head(10)
)

most_underpaid_relative_to_model[[
    "player",
    "team",
    "group",
    "actual",
    "predicted",
    "residual",
    "abs_error"
]]

,player,team,group,actual,predicted,residual,abs_error
54,Odyssey Sims,TOT,veteran,7949,138897.386595,130948.386595,130948.386595
53,Odyssey Sims,TOT,veteran,13911,138897.386595,124986.386595,124986.386595
104,Kaila Charles,TOT,veteran,13911,113460.868377,99549.868377,99549.868377
103,Kaila Charles,TOT,veteran,13911,113460.868377,99549.868377,99549.868377
91,Haley Jones,TOT,veteran,36094,128767.144015,92673.144015,92673.144015
102,Kaila Charles,TOT,veteran,21198,113460.868377,92262.868377,92262.868377
153,Zia Cooke,SEA,veteran,13882,101078.034459,87196.034459,87196.034459
52,Odyssey Sims,TOT,veteran,54622,138897.386595,84275.386595,84275.386595
57,Aari McDonald,IND,veteran,52333,130636.112736,78303.112736,78303.112736
37,Veronica Burton,GSV,veteran,103831,181619.911208,77788.911208,77788.911208


In [26]:
# Most overpaid relative to the model-predicted salary:
# predicted salary is much lower than actual salary. The smaller the residual, the more overpaid the player is relative to the model's prediction.
most_overpaid_relative_to_model = (
    pred_df
    .sort_values("residual", ascending=True)
    .head(10)
)

most_overpaid_relative_to_model[[
    "player",
    "team",
    "group",
    "actual",
    "predicted",
    "residual",
    "abs_error"
]]

,player,team,group,actual,predicted,residual,abs_error
194,Moriah Jefferson,CHI,veteran,145500,22577.850691,-122922.149309,122922.149309
9,Sabrina Ionescu,NYL,rookie,222060,106060.957794,-115999.042206,115999.042206
112,Teaira McCowan,DAL,veteran,201400,89401.316216,-111998.683784,111998.683784
16,Arike Ogunbowale,DAL,unknown,249032,139530.215438,-109501.784562,109501.784562
44,Jewell Loyd,LVA,veteran,249032,140885.526888,-108146.473112,108146.473112
144,Alysha Clark,TOT,veteran,205908,118245.526723,-87662.473277,87662.473277
186,Shatori Walker-Kimbrough,ATL,veteran,150000,64335.455415,-85664.544585,85664.544585
59,Brittney Griner,ATL,veteran,214466,129746.721674,-84719.278326,84719.278326
158,Karlie Samuelson,MIN,veteran,118450,34809.561242,-83640.438758,83640.438758
81,Myisha Hines-Allen,DAL,veteran,203000,121782.995450,-81217.004550,81217.004550


### Player-level audit notes

The player-level error analysis shows that the largest errors occur when the model prediction differs substantially from the observed 2025 salary. Some large errors involve low actual salaries with much higher predicted salaries, which helps explain why the overall MAPE is high since when the actual salary is small, a large dollar error becomes an extremely large percentage error.

Using `residual = predicted - actual`, large positive residuals indicate players who appear underpaid relative to the model-predicted salary, while small negative residuals indicate players who appear overpaid relative to the model-predicted salary.

**These results should be interpreted as model-relative signals, not definitive claims about whether a player is truly underpaid or overpaid. Some players also appear multiple times, so the unit of analysis should be checked before making strong player-level conclusions.**


## 6. Save audit summary tables

This section saves the key audit tables generated in this notebook.


In [28]:
# Save key audit summary tables generated in this notebook.

audit_dir = result_dir / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)

top_tuning_path = audit_dir / "top_tuning_audit.csv"
top_tuning.to_csv(top_tuning_path, index=False)

metrics_path = audit_dir / "holdout_metrics_audit.csv"
metrics_df.to_csv(metrics_path, index=False)

largest_errors_path = audit_dir / "largest_prediction_errors_audit.csv"
largest_abs_errors.to_csv(largest_errors_path, index=False)

underpaid_path = audit_dir / "most_underpaid_relative_to_model_audit.csv"
most_underpaid_relative_to_model.to_csv(underpaid_path, index=False)

overpaid_path = audit_dir / "most_overpaid_relative_to_model_audit.csv"
most_overpaid_relative_to_model.to_csv(overpaid_path, index=False)

print(f"Saved top tuning table to: {top_tuning_path}")
print(f"Saved holdout metrics to: {metrics_path}")
print(f"Saved largest prediction errors to: {largest_errors_path}")
print(f"Saved most underpaid relative to model table to: {underpaid_path}")
print(f"Saved most overpaid relative to model table to: {overpaid_path}")


Saved top tuning table to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\audit\top_tuning_audit.csv
Saved holdout metrics to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\audit\holdout_metrics_audit.csv
Saved largest prediction errors to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\audit\largest_prediction_errors_audit.csv
Saved most underpaid relative to model table to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\audit\most_underpaid_relative_to_model_audit.csv
Saved most overpaid relative to model table to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\audit\most_overpaid_relative_to_model_audit.csv


## 7. Final model comparison conclusion

The final selected model is the tuned Random Forest model.

Key audit conclusions:

* **Model selection:** Random Forest is a reasonable final choice under the RMSE-first rule. It performed best in the baseline evaluation and also achieved the lowest cross-validated RMSE among the tuned candidate models.

* **Baseline comparison:** The broader baseline comparison included Random Forest, Ridge, Linear Regression, and Dummy Regressor. These results are saved in `baseline_evaluation.csv`.

* **Tuning scope:** The tuning stage focused only on Random Forest and Gradient Boosting. Therefore, the final tuned model should be interpreted as the best model within this tuning scope, not necessarily the best possible model across all model families.

* **CV variability:** The best Random Forest model only improves CV RMSE over the best Gradient Boosting model by about 279.53. This difference is small compared with the `std_test_score` values, which are about 3,517.12 for Random Forest and 3,932.08 for Gradient Boosting. This means the exact ranking between the top tuned models should not be over-interpreted.

* **Generalization gap:** Random Forest has a larger train-validation gap than Gradient Boosting, suggesting possible overfitting. 

* **2025 holdout performance:** On the 2025 holdout set, the final Random Forest model has reasonable RMSE and MAE performance.

* **MAPE interpretation:** The aggregate MAPE is high and unstable because percentage error is sensitive to small actual salary values. Therefore, the final evaluation should emphasize **RMSE, MAE, and player-level residual analysis**.
